In [6]:
import importlib
import pandas as pd
import numpy as np
import re
from pathlib import Path

import functions
import functions_GLM
importlib.reload(functions)
importlib.reload(functions_GLM)
from functions import *
from functions_GLM import *

df_1111 = pd.read_csv("AI_Lit_Que_1111.csv")
df_1204 = pd.read_csv("AI_Lit_Que_1204.csv")

# add source/course label
df_1111["course"] = "1111"
df_1204["course"] = "1204"

print(df_1111.shape, df_1204.shape)

df, meta = prepare_dataset("AI_Lit_Que_1111.csv", "AI_Lit_Que_1204.csv")


(118, 37) (23, 37)


# Generalized linear model
$$
M_i = \beta_0 + \beta_1 A_i + \boldsymbol{\beta}_c^\top C_i + \varepsilon_i
$$

$$
\log\{P(Y_i=1 \mid A_i, M_i, C_i)\}
= \theta_0 + \theta_1 A_i + \theta_2 M_i + \boldsymbol{\theta}_c^\top C_i
$$

where $Y_i$ is the binary AI literacy(High or low) variable, $A_i$ is the SES variable, $M_i$ is the mediator variable.


In [ ]:
# add mediator
df_analysis = add_mediator_composites(df, mediator_map)

#efa
ai_efa_items = [
    "ai_concept_data_bias_scored_num",
    "ai_concept_blackbox_scored_num",
    "ai_concept_input_variation_scored_num",
    "ai_concept_prompt_wording_scored_num",
    "ai_concept_social_ethics_scored_num",
    "ai_ability_training_data_scored_num",
    "ai_ability_explainability_scored_num",
    "ai_ability_input_sensitivity_scored_num",
    "ai_ability_prompting_scored_num",
    "ai_ability_social_ethics_scored_num",
]

fa_combined, d_combined, loadings_combined, variance_combined = fit_efa(
    df=df,
    sample="Combined",
    items=ai_efa_items,
    n_factors=2,
    rotation="oblimin"
)

scores = fa_combined.transform(d_combined)
df_factors = pd.DataFrame(
    scores,
    index=d_combined.index,
    columns=["ai_factor1_score", "ai_factor2_score"]
)

if "ai_factor1_score" in df_factors.columns and "ai_factor2_score" in df_factors.columns:
    df_analysis.loc[df_factors.index, "ai_factor1_score"] = df_factors["ai_factor1_score"]
    df_analysis.loc[df_factors.index, "ai_factor2_score"] = df_factors["ai_factor2_score"]

df_bin, cut1 = make_binary_outcome(
    df_analysis,
    source_col="ai_factor1_score",
    new_col="ai_factor1_high",
    threshold="median",
    higher_is_one=True
)

print("Cutoff used:", cut1)
df_bin["ai_factor1_high"].value_counts(dropna=False)


Cutoff used: 0.0002898421999757107


1    71
0    70
Name: ai_factor1_high, dtype: int64

In [ ]:
mediators = [
    "language_load_score",
    "epistemic_stance_score",
    "practical_ai_use_score",
    "learning_ecology_score",
    "conceptual_exposure_score"
]

poisson_results = run_poisson_mediations(
    df=df_bin,
    x="ses_index",
    mediators=mediators,
    y_bin="ai_factor1_high",
    sample="Combined",
    covariates=[],
    n_boot=2000,
    seed=2026
)

poisson_results.round(3)